In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_04_events_mapping
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_04_events_mapping
# Spec Ref  : §1.4 Events Mapping (events_raw → events)
# Purpose   : Map all events from all sources to resolved contact IDs.
#             Unions BQ events (hgi.silver.events) + SF crm_events
#             into one mapped_events table.
#
# Stage-Swap pattern (spec §1.4):
#   1. Write to mapped_events_stage first
#   2. CREATE OR REPLACE mapped_events from stage
#   3. Drop stage
#   Ensures zero downtime — mapped_events is always queryable.
#
# Contact resolution (spec §1.4 — LEFT JOIN):
#   events_raw.source_key_value contains composite ID (bigquery_Event_event_id_...)
#   We resolve to contacts_all.id via two attempts:
#     1. Composite ID match: c.id = concat(source_system, '_Contact_Id_', source_key_value)
#     2. Email match: lower(c.email) = lower(source_key_value)
#   NULL contactid = anonymous event — allowed and expected per spec
#
# Inputs  : hgi.silver.events (BQ), hgi.silver.crm_events (SF Tasks), hgi.silver.contacts_all
# Output  : hgi.silver.mapped_events
#
# Run after: map_01 (contacts_all), map_02 (accounts_all), map_03 (crm_events)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2 ── Widget
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
tenant_id   = tenant_id_from_customer(customer_id)

print(f"=== Map 04: events mapping (events_raw → mapped_events) ===  customer={customer_id}  tenant={tenant_id}")
print(f"  BQ source  : {sv}.events")
print(f"  SF source  : {sv}.crm_events")
print(f"  Output     : {sv}.mapped_events")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"

import sys, os
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
if os.path.abspath(project_root) not in sys.path:
    sys.path.append(os.path.abspath(project_root))

from utils.pipeline_metrics import PipelineMetrics
pm = PipelineMetrics(
    spark          = spark,
    parent_run_id  = parent_run_id,
    job_name       = "02b_map_04_events_mapping",
    task_key       = "run_job_D_silver_map",
    source_system  = source_system,
    load_type      = load_type,
    customer_id    = customer_id,
    object_name    = object_name,
)
pm.init()

initialize_audit_tables()

In [0]:
# CELL 3 ── Incremental watermark MERGE for mapped_events (tenant-scoped)
# Only processes events newer than the last watermark — avoids re-mapping ALL events.
# Events are append-only, so no WHEN NOT MATCHED BY SOURCE (no deletes needed).
try:
    # Safe drop in case target exists as a VIEW
    safe_drop_view(f"{sv}.mapped_events")

    # Ensure table exists with data_timestamp for watermark tracking
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {sv}.mapped_events (
            id              STRING,
            tenant          BIGINT,
            event_timestamp TIMESTAMP,
            meta_event      STRING,
            contactid       STRING,
            domain          STRING,
            event_text      STRING,
            data_timestamp  TIMESTAMP
        )
        USING DELTA
        CLUSTER BY (tenant, meta_event)
        {DELTA_TBLPROPS_MAP}
    """)

    # Add data_timestamp column if missing (existing table from before this change)
    existing_cols = [c.name.lower() for c in spark.table(f"{sv}.mapped_events").schema]
    if "data_timestamp" not in existing_cols:
        spark.sql(f"ALTER TABLE {sv}.mapped_events ADD COLUMN data_timestamp TIMESTAMP")
        print("  Added data_timestamp column to mapped_events")

    # Safety: ensure crm_events has data_timestamp column
    # map_03 adds it, but map_03 and map_04 run in PARALLEL in the job
    try:
        crm_cols = [c.name.lower() for c in spark.table(f"{sv}.crm_events").schema]
        if "data_timestamp" not in crm_cols:
            spark.sql(f"ALTER TABLE {sv}.crm_events ADD COLUMN data_timestamp TIMESTAMP")
            print("  Added data_timestamp column to crm_events (safety: map_03 runs in parallel)")
    except Exception as alt_err:
        print(f"  Note: crm_events ALTER TABLE skipped: {alt_err}")

    # Watermark: only process events newer than last processed for this tenant
    last_ts = spark.sql(f"""
        SELECT COALESCE(MAX(data_timestamp), TIMESTAMP('1900-01-01'))
        FROM {sv}.mapped_events
        WHERE tenant = {tenant_id}
    """).collect()[0][0]
    print(f"  Watermark for tenant {tenant_id}: {last_ts}")

    spark.sql(f"""
        MERGE INTO {sv}.mapped_events AS target
        USING (
            -- ── Part A: BQ native events (new since watermark) ──────────
            SELECT
                e.id, e.tenant,
                CAST(e.event_timestamp AS TIMESTAMP) AS event_timestamp,
                COALESCE(
                    LOWER(REGEXP_REPLACE(e.event, '[^a-zA-Z0-9]+', '_')),
                    'unknown_event'
                ) AS meta_event,
                COALESCE(c1.id, c2.id) AS contactid,
                e.domain,
                e.event_text,
                e.data_timestamp

            FROM {sv}.events e
            LEFT JOIN {sv}.contacts_all c1
                ON c1.id = CONCAT(e.source_system, '_Contact_Id_', e.source_key_value)
            LEFT JOIN {sv}.contacts_all c2
                ON LOWER(c2.email) = LOWER(e.source_key_value)
                AND c1.id IS NULL
            WHERE e.tenant = {tenant_id}
              AND e.data_timestamp > '{last_ts}'

            UNION ALL

            -- ── Part B: SF CRM events (new since watermark) ───────────
            SELECT
                ce.id, ce.tenant,
                ce.event_timestamp,
                ce.meta_event,
                COALESCE(c.id, ce.contact_id) AS contactid,
                c.domain,
                ce.event_text,
                COALESCE(ce.data_timestamp, ce.event_timestamp) AS data_timestamp

            FROM {sv}.crm_events ce
            LEFT JOIN {sv}.contacts_all c
                ON c.id = ce.contact_id
            WHERE ce.tenant = {tenant_id}
              AND COALESCE(ce.data_timestamp, ce.event_timestamp) > '{last_ts}'
        ) AS source
        ON target.id = source.id AND target.tenant = source.tenant
        WHEN MATCHED THEN UPDATE SET
            target.event_timestamp = source.event_timestamp,
            target.meta_event      = source.meta_event,
            target.contactid       = source.contactid,
            target.domain          = source.domain,
            target.event_text      = source.event_text,
            target.data_timestamp  = source.data_timestamp
        WHEN NOT MATCHED THEN INSERT (id, tenant, event_timestamp, meta_event,
            contactid, domain, event_text, data_timestamp)
            VALUES (source.id, source.tenant, source.event_timestamp, source.meta_event,
                source.contactid, source.domain, source.event_text, source.data_timestamp)
    """)

    new_count = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.mapped_events WHERE tenant = {tenant_id}
          AND data_timestamp > '{last_ts}'
    """).collect()[0][0]
    print(f"  Incremental: {new_count:,} new/updated events merged for tenant {tenant_id}")
except Exception as e:
    print(f"\u274c mapped_events build failed: {e}")
    log_audit(
        job_name       = "02b_map_04_events_mapping",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
# CELL 4 ── Cleanup stage table (no longer used but may exist from previous runs)
try:
    spark.sql(f"DROP TABLE IF EXISTS {sv}.mapped_events_stage")
    print(f"  mapped_events stage table cleaned up (tenant-scoped rebuild used instead)")
except Exception as e:
    print(f"⚠ Stage cleanup warning (non-fatal): {e}")

In [0]:
# CELL 5 ── Spec DQ verification
try:
    total        = cnt(f"{sv}.mapped_events")
    with_contact = spark.sql(
        f"SELECT COUNT(*) FROM {sv}.mapped_events WHERE contactid IS NOT NULL"
    ).collect()[0][0]
    null_meta    = spark.sql(
        f"SELECT COUNT(*) FROM {sv}.mapped_events WHERE meta_event IS NULL"
    ).collect()[0][0]

    contact_pct  = round(100 * with_contact / max(total, 1), 1)
    meta_pct     = round(100 * (total - null_meta) / max(total, 1), 1)

    print(f"  mapped_events: {total:,} rows")
    print(f"  Contact resolution: {with_contact:,}/{total:,} = {contact_pct}%")
    print(f"    Spec gate: ≥{DQ_THRESHOLDS['event_contact_resolution_pct']}%  {'✅ PASS' if contact_pct >= DQ_THRESHOLDS['event_contact_resolution_pct'] else '⚠ BELOW THRESHOLD'}")
    print(f"  meta_event coverage: {meta_pct}%")
    print(f"    Spec gate: 100%  {'✅ PASS' if null_meta == 0 else '⚠ BELOW THRESHOLD'}")
except Exception as e:
    print(f"❌ DQ verification failed: {e}")
    log_audit(
        job_name       = "02b_map_04_events_mapping",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "DQ_CHECK_FAILED",
        error_type     = type(e).__name__,
        error_reason   = f"DQ check failed: {str(e)[:450]}",
    )
    raise

In [0]:
try:
    total_rows = spark.table(f"{sv}.mapped_events").count()
    pm.save(status="SUCCESS", rows_processed=total_rows)
except Exception as e:
    print(f"❌ Metrics save failed: {e}")
    log_audit(
        job_name       = "02b_map_04_events_mapping",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise